**Assignment:** Suppose that you have been hired as a consultant to assess the impact of two controversial changes that a physician specialty board implemented in 1990. Physicians are not required to obtain board certification for this specialty to practice medicine, but the majority of doctors in this specialty do take the exam upon completing their residency program. If a physician fails to pass the exam, they have the option of re-taking the exam the following year or any year after that. This specialty board maintains a performance monitoring program where Physicians are rated on a 100 point scale based on how closely their practice patterns align with a variety of published clinical guidelines that the board endorses. There is strong supporting evidence that this composite score is positively correlated with patient outcomes -- in other words, physicians who score highly according to this metric are likely to provide patients with better care than those who score poorly. This data is only available for certified physicians.
In 1990, this specialty board made the decision to revamp their certification process in an attempt to increase the quality of care delivered by their physicians. Two primary changes were made:
First, a human-administered test component was added to the initial certification exams, in which physicians seeking to obtain certification must pass an oral interview administered by certified board examiner. This was intended to make the certification exams more stringent.
Second, a requirement was added that certified physicians must complete maintenance-of-certification exams every five years in order maintain their certification status. Previously, physicians who passed the initial exam received lifetime certification status. The goal of this change was to increase physicians' awareness of clinical best practices and the corresponding guideline-adherence scores in the attached data set.
However, the changes were highly controversial among physicians, many of whom felt that the additional requirements would be too burdensome and would be no more effective at increasing quality of care than the previous certification process.

**Questions:**
1. Prior to 1990, to what degree does it appear that physicians who passed their initial certification exams were more/less likely to follow clinical guidelines during their subsequent careers?
2. How do the two changes that were made to the certification process in 1990 appear to have impacted physician behavior? Does the board appear to have succeeded in making the initial certification exams more stringent? Have the maintenance-of-certification exams increased guideline adherence? Were physicians' concerns about the changes justified? In responding, please note any instances where you think of multiple hypotheses that might explain a given observation.

**Data Description:** To answer this question, its given the attached spreadsheet, which contains data on every physician who received board certification for this specialty between 1970 and 2002. In the background tab of the spreadsheet, its found the year that each physician completed his/her residency training and the year that s/he received initial board certification. In the adherence_evaluations tab - annual results of a performance monitoring program that this specialty board maintains.
The attached data does not explicitly indicate whether a physician passed or failed their certification exam(s). However, prior analysis of the 1970-1990 data set has shown that all physicians who obtained certification the same year they completed residency passed their board exams on the first attempt. On the contrary, during the same time period, most physicians who obtained certification in subsequent years after completing residency did so because they failed the board exams the first year they took them.

DATA LOADING

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl

data = pd.read_excel('match_data_exercise_to_share.xlsx', sheet_name=None)

xlsx = pd.ExcelFile('match_data_exercise_to_share.xlsx')
bg = xlsx.parse('background')
eval = xlsx.parse('adherence_evaluations')

print(bg)
print(eval)

       physician_uid  training_year  board_cert_year
0        10000831640           1970             1970
1        10001811121           1970             1970
2        10002674776           1970             1970
3        10003837524           1970             1970
4        10004856810           1970             1970
...              ...            ...              ...
15467    25994322770           2001             2001
15468    25995446298           2001             2001
15469    25996388376           2001             2001
15470    25997239841           2001             2001
15471    25998368317           2001             2001

[15472 rows x 3 columns]
          Unnamed: 0  evaluation_year   Unnamed: 2   Unnamed: 3   Unnamed: 4  \
0      physician_uid      1971.000000  1972.000000  1973.000000  1974.000000   
1        10000831640        45.558857    56.595450    53.878878    36.174120   
2        10001811121        49.955430    80.417479    97.212223    74.138971   
3        100026747

EDA

In [ ]:
print(bg.info()) #data structure
print(bg.head()) #first 5 rows

print(eval.info()) #data structure
print(eval.head()) #first 5 rows

new_cols = eval.iloc[0].values
new_cols[0] = 'physician_uid'
eval_clean = eval.drop(index=0).copy()
eval_clean.columns = new_cols

df_evals = eval_clean.melt(id_vars='physician_uid', var_name='evaluation_year', value_name='adherence_score')

df_evals['physician_uid'] = pd.to_numeric(df_evals['physician_uid'])
df_evals['adherence_score'] = pd.to_numeric(df_evals['adherence_score'], errors='coerce')
df_evals['evaluation_year'] = pd.to_numeric(df_evals['evaluation_year'], errors='coerce')

df_evals = df_evals.dropna(subset=['adherence_score'])
df = pd.merge(eval_long.dropna(), bg, on='physician_uid', how='inner')

df['passed_immediately'] = (df['board_cert_year'] == df['training_year']).astype(int)
bg['passed_immediately'] = (bg['board_cert_year'] == bg['training_year']).astype(int)

df['era'] = np.where(df['board_cert_year'] < 1990, 'Pre-1990', 'Post-1990')
bg['era'] = np.where(bg['board_cert_year'] < 1990, 'Pre-1990', 'Post-1990')

df['years_since_cert'] = df['evaluation_year'] - df['board_cert_year']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15472 entries, 0 to 15471
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   physician_uid       15472 non-null  int64 
 1   training_year       15472 non-null  int64 
 2   board_cert_year     15472 non-null  int64 
 3   era                 15472 non-null  object
 4   passed_immediately  15472 non-null  int64 
dtypes: int64(4), object(1)
memory usage: 604.5+ KB
None
   physician_uid  training_year  board_cert_year       era  passed_immediately
0    10000831640           1970             1970  Pre-1990                   1
1    10001811121           1970             1970  Pre-1990                   1
2    10002674776           1970             1970  Pre-1990                   1
3    10003837524           1970             1970  Pre-1990                   1
4    10004856810           1970             1970  Pre-1990                   1
<class 'pandas.core.frame

1. Prior to 1990, to what degree does it appear that physicians who passed their initial certification exams were more/less likely to follow clinical guidelines during their subsequent careers?

In [ ]:
pre_90_analysis = df[df['evaluation_year'] < 1990].groupby('passed_immediately')['adherence_score'].mean()

print("Q1 Analysis: Pre-1990 Quality (0 = Retaker, 1 = First Pass)")
print(pre_90_analysis)
# Conclusion: Physicians who passed on the first attempt scored ~10 points higher (69.4 vs 59.9).
# This indicates that the original certification was already a strong predictor of clinical quality.

Q1 Analysis: Pre-1990 Quality (0 = Retaker, 1 = First Pass)
passed_immediately
0    59.862175
1    69.382102
Name: adherence_score, dtype: float64


2. How do the two changes that were made to the certification process in 1990 appear to have impacted physician behavior? Does the board appear to have succeeded in making the initial certification exams more stringent? Have the maintenance-of-certification exams increased guideline adherence? Were physicians' concerns about the changes justified? In responding, please note any instances where you think of multiple hypotheses that might explain a given observation.

In [ ]:
stringency = bg.groupby('era')['passed_immediately'].mean()

print("Certification Pass Rate (Stringency):", stringency)
# The immediate pass rate dropped from ~94.6% to ~72.6%.
# The board successfully made the initial certification more stringent.

moc_trend = df[df['era'] == 'Post-1990'].groupby('years_since_cert')['adherence_score'].mean().head(12)
print("MOC Impact on Post-1990 Cohort:", moc_trend)
# Scores show a slight decline by year 5 (68.6) followed by a rebound in year 6 (69.7).
# This suggests the 5-year MOC cycle acts as a "refresher," forcing physicians to update their knowledge.

Certification Pass Rate (Stringency): era
Post-1990    0.726117
Pre-1990     0.946117
Name: passed_immediately, dtype: float64
MOC Impact on Post-1990 Cohort: years_since_cert
1.0     69.781618
2.0     69.351652
3.0     69.194203
4.0     69.472934
5.0     68.635367
6.0     69.709317
7.0     70.060515
8.0     70.136043
9.0     69.810318
10.0    69.848934
11.0    67.658139
12.0    67.954898
Name: adherence_score, dtype: float64


EXPORT

In [ ]:
medical_cleaned_data= df[[
    'physician_uid', 'evaluation_year', 'adherence_score',
    'board_cert_year', 'training_year',
    'passed_immediately', 'years_since_cert', 'era'
]]

medical_cleaned_data.to_csv('medical_cleaned_data', index=False)